Import dữ liệu từ kaggle

In [1]:
import kagglehub
path = kagglehub.dataset_download("minhtruongngoc/vn-reviews-data")
path

100%|██████████| 665k/665k [00:00<00:00, 101MB/s]

Extracting files...


'/root/.cache/kagglehub/datasets/minhtruongngoc/vn-reviews-data/versions/1'

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [3]:
# Set cố định random seed
np.random.seed(42)
tf.random.set_seed(42)

Đọc dữ liệu

In [4]:
pos_df = pd.read_csv("/root/.cache/kagglehub/datasets/minhtruongngoc/vn-reviews-data/versions/1/positive_data.csv")
pos_df.head(5)

,Rate,Review,Label
0,9.0,Khu ẩm thực với đa dạng đồ lại còn bày trí đẹ...,1
1,9.0,Lúc nào đến aeon là lúc đấy phải tống một đốn...,1
2,10.0,Bánh ngon lại rẻ chê đâu được gần hết các loạ...,1
3,9.0,Ngon rẻ,1
4,9.6,Tôi sắp chết vì ngập trong sushi mấttttt Lên ...,1


In [5]:
neg_df = pd.read_csv("/root/.cache/kagglehub/datasets/minhtruongngoc/vn-reviews-data/versions/1/negative_data.csv")
neg_df.head(5)

,Rate,Review,Label
0,4.0,Mình thề là mình ko thể cảm nổi đồ ăn ở aeon ...,-1
1,3.8,Đôi khi thèm lên là bất chấp nắng nóng phi Và...,-1
2,3.8,Ngõ treo biển cafe trứng đúng kiểu phố cổ hà ...,-1
3,3.8,Mình thấy địa chỉ cafe Giảng ở Nguyễn Hữu Huâ...,-1
4,2.2,Mình là người Hà Nội và cũng cực kỳ khó tính ...,-1


Số giá trị (số dòng) positive và negative

In [6]:
pos_df['Label'].value_counts()

,count
Label,
1,2642


In [7]:
neg_df["Label"].value_counts()

,count
Label,
-1,1765


Trong một số trường hợp, số lượng samples của các class có tỉ lệ quá khác biệt sẽ ảnh hưởng đến chất lượng của mô hình. Vì vậy nên sử dụng phương thức chọn mẫu DataFrame.sample() để cho số lượng mẫu positive bằng với số lượng mẫu negative.




In [8]:
pos_df_sampled = pos_df.sample(n=len(neg_df), random_state=42)

Kết hợp dữ liệu positive và negative vào cùng một DataFrame bằng pandas.concat

In [9]:
df = pd.concat([pos_df_sampled, neg_df])
df['Label'].value_counts()

,count
Label,
1,1765
-1,1765


-  pandas.concat() làm cho sample positive và negative hiện đang không được phân bố đồng đều
- Để phân phối lại các sample positive và negative trong df, sử dụng phương thức DataFrame.sample.
- Ở đây ta chọn frac=1 để chọn tất cả các sample trong df với thứ tự ngẫu nhiên.

In [10]:
df = df.sample(frac=1).reset_index(drop=True)
df.head()

,Rate,Review,Label
0,9.0,Nhà mình người đi ăn mà hóa đơn là đắt hay rẻ...,1
1,3.4,Mình nói thật mình ăn ở đây nhiều rồi nhưng h...,-1
2,2.8,Đặc sệt hơi hướng nhà Nhân viên thì đông như ...,-1
3,9.0,Đến lần k phải giờ cao điểm nên khá thoải mái...,1
4,9.4,Soya bean ăn chất lượng,1


- Đầu ra của mô hình phân loại hai lớp (binary classification) là giá trị 0 hoặc 1.
- Vì vậy cần chuyển các giá trị -1 thành 0 bằng DataFrame.replace.

In [11]:
# DataFrame.replace() nhận vào một dictionary
# với key là giá trị cần thay đổi và value là giá trị mới
df['Label'] = df['Label'].replace({-1:0})
df['Label'].value_counts()

,count
Label,
1,1765
0,1765


### **Chia tập train - test**

- Sử dụng thuộc tính values của Pandas Series để lấy các giá trị dưới dạng numpy ndarray.
- Các dữ liệu dùng để train và test nên được chuyển đổi sang kiểu ndarray để phù hợp với các hàm tính toán trong Machine Learning và Deep Learning.

In [12]:
X = df['Review'].values
y = df['Label'].values
X.shape, y.shape

((3530,), (3530,))

Sử dụng hàm train_test_split của sklearn để chia tập train và test theo tỉ lệ 8:2.




In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = np.array(X_train)
X_test = np.array(X_test)
y_train = np.array(y_train)
y_test = np.array(y_test)
X_train.shape, X_test.shape

((2824,), (706,))

### **Tokenize dữ liệu train - test**

- Vận dụng các kiến thức tokenizing và padding ở buổi trước để xử lý dữ liệu chữ.
- Nhắc lại các bước xử lý:
1. Chọn kích thước từ điển
2. Chọn độ dài lớn nhất của một chuỗi (sequence)
3. Tạo Tokenizer và fit Tokenizer (trên văn bản của tập train)
4. Sử dụng Tokenizer đã huấn luyện để tạo tokens
5. Sử dụng padding và truncating để các chuỗi có độ dài bằng nhau

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [15]:
# Bước 1: Chọn kích thước từ điển là 10000
vocab_size = 10000

In [16]:
# Bước 2: Độ dài lớn nhất của một bình luận là 400 tokens
max_length = 400

In [18]:
# Bước 3: Tạo Tokenizer và fit Tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [20]:
# Bước 4: Sử dụng Tokenizer đã huấn luyện để tạo tokens
# Bước 5: Sử dụng padding và truncating để các chuỗi có độ dài bằng nhau

# Sử dụng tokenizer đã fit để tokenize các bình luận trên tập train
X_train_seqs = tokenizer.texts_to_sequences(X_train)
X_train_padded = pad_sequences(X_train_seqs, maxlen=max_length, padding='post', truncating='post') # thực hiện padding

# Sử dụng tokenizer đã fit để tokenize các bình luận trên tập test
X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_test_padded = pad_sequences(X_test_seqs, maxlen=max_length, padding='post', truncating='post') # thực hiện padding

In [21]:
print("Length of train dataset:", len(X_train_padded))
print("Length of test dataset:", len(X_test_padded))

Length of train dataset: 2824
Length of test dataset: 706


### **Định nghĩa mô hình NLP**

Mô hình phân loại ngôn ngữ đơn giản gồm 2 phần:
1. Layer Embedding: Để học ngôn ngữ
2. Block phân loại: Tương tự với mô hình phân loại hình ảnh

Vì mục tiêu bài toán là phân loại 02 lớp nên cần sử dụng loss là `binary_crossentropy` và metric là `accuracy`.\
Mặc định sử dụng optimizer là `adam`.

In [22]:
# Chọn chiều embedding là 128
embedding_dim = 128

model = keras.Sequential([
    # Layer Embedding
    keras.layers.Embedding(vocab_size, embedding_dim, trainable=True), # vocab_size=10000, max_length=400
    # Layer phân loại (đã học)
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])       # mô hình phân loại thường sử dụng thang đo là acc
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Sử dụng thêm ModelCheckpoint và chạy phương thức `fit` để huấn luyện mô hình.

In [23]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath='model_epoch_{epoch:02d}.keras',
    save_weights_only=False,
    save_best_only=False,
    monitor='val_loss',
    verbose=1
)

# Huấn luyện mô hình với config đã chọn
history = model.fit(
    X_train_padded, y_train,
    validation_data=(X_test_padded, y_test),
    epochs=8,                            # Số lần train toàn bộ dữ liệu
    callbacks=[checkpoint]               # Thêm config checkpoint
)

Epoch 1/8
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.5275 - loss: 0.7577
Epoch 1: saving model to model_epoch_01.keras

Epoch 1: finished saving model to model_epoch_01.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 8s 71ms/step - accuracy: 0.5662 - loss: 0.7212 - val_accuracy: 0.5425 - val_loss: 0.6435
Epoch 2/8
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.6517 - loss: 0.6000
Epoch 2: saving model to model_epoch_02.keras

Epoch 2: finished saving model to model_epoch_02.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 7s 74ms/step - accuracy: 0.7295 - loss: 0.5252 - val_accuracy: 0.8541 - val_loss: 0.3688
Epoch 3/8
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.8914 - loss: 0.2794
Epoch 3: saving model to model_epoch_03.keras

Epoch 3: finished saving model to model_epoch_03.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 6s 67ms/step - accuracy: 0.9051 - loss: 0.2497 - val_accuracy: 0.8484 - val_loss: 0.3445
Epoch 4/8
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9473 - loss: 0.1499
Epoch 4: s

### **Test mô hình**
- Hãy tạo một số dữ liệu mẫu để đánh giá mô hình.
- Ở đây ta có một số bình luận mẫu `test_feedback` và giá trị phân loại tương ứng `test_truth`.\

In [24]:
test_reviews = [
    "Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn.",
    "Đồ ăn k giống review trên mạng, chất lượng tệ, không ngon. Tốn tiền!",
    "Rất thích không gian quán",
    "Không thể tin được phải đi 5km để ăn một món như thế này",
    "Không thể tin được chỉ cần đi 5km để ăn một món như thế này",
    "Tôi không ghét món dưa leo lắm",
    "Tôi không ghét món dưa leo lắm. Ăn cũng được",
    "Đồ ăn nhiều bột ngọt. Không ăn nữa",
    "Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. Tuy nhiên không gian quán đẹp, sẽ quay lại lần sau"
]
test_gt = [0, 0, 1, 0, 1, 1, 1, 0, 1]
test_df = pd.DataFrame({"Review": test_reviews, "Label": test_gt})
test_df

,Review,Label
0,Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn.,0
1,"Đồ ăn k giống review trên mạng, chất lượng tệ,...",0
2,Rất thích không gian quán,1
3,Không thể tin được phải đi 5km để ăn một món n...,0
4,Không thể tin được chỉ cần đi 5km để ăn một mó...,1
5,Tôi không ghét món dưa leo lắm,1
6,Tôi không ghét món dưa leo lắm. Ăn cũng được,1
7,Đồ ăn nhiều bột ngọt. Không ăn nữa,0
8,"Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. T...",1


Các bước sử dụng mô hình để phân loại:
1. Sử dụng tokenizer đã fit ở bước trước để tokenize các bình luận
2. Thực hiện padding tương ứng
3. Dự đoán bằng phương thức `predict` với mô hình
4. Xử lý kết quả dự đoán

In [25]:
# Sử dụng lại tokenizer
test_reviews = test_df['Review'].values
test_labels = test_df['Label'].values
test_seqs = tokenizer.texts_to_sequences(test_reviews)
test_padded = pad_sequences(test_seqs, padding='post', truncating='post', maxlen=max_length) # thực hiện padding

In [26]:
from tensorflow.keras.models import load_model

loaded_model = load_model("model_epoch_04.keras")
preds = loaded_model.predict(test_padded)
preds

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


array([[0.00877251],
       [0.00731593],
       [0.8700556 ],
       [0.05141821],
       [0.03695592],
       [0.16142724],
       [0.15585285],
       [0.07678194],
       [0.46102026]], dtype=float32)

In [27]:
classes = {"1":"positive", "0": "negative"}
acc = 0
i = 0
threshold = 0.5

for fb, score in zip(test_reviews, preds):
    prediction = (score>threshold).astype(int)
    print(fb, "|", classes[str(test_labels[i])], "predicted as", classes[str(prediction[0])], "with score =", score[0])

    if prediction == test_labels[i]:
        acc += 1
    i += 1

print("\nAccuracy:", round(acc/len(test_reviews), 2))

Thịt bị hôi. Nước bún thì nhạt. Đề nghị đừng ăn. | negative predicted as negative with score = 0.008772512
Đồ ăn k giống review trên mạng, chất lượng tệ, không ngon. Tốn tiền! | negative predicted as negative with score = 0.007315933
Rất thích không gian quán | positive predicted as positive with score = 0.8700556
Không thể tin được phải đi 5km để ăn một món như thế này | negative predicted as negative with score = 0.051418208
Không thể tin được chỉ cần đi 5km để ăn một món như thế này | positive predicted as negative with score = 0.036955923
Tôi không ghét món dưa leo lắm | positive predicted as negative with score = 0.16142724
Tôi không ghét món dưa leo lắm. Ăn cũng được | positive predicted as negative with score = 0.15585285
Đồ ăn nhiều bột ngọt. Không ăn nữa | negative predicted as negative with score = 0.07678194
Nhiều đồ ăn, khá ngon. Nhưng giá cả hơi mắc. Tuy nhiên không gian quán đẹp, sẽ quay lại lần sau | positive predicted as negative with score = 0.46102026

Accuracy: 0.56
